# Ensemble Model - LightGBM + XGBoost

This notebook tests an ensemble approach combining the two best-performing algorithms.

## Objective

Create a weighted ensemble of LightGBM and XGBoost to potentially improve detection performance.

## Approach

- Soft voting with probability averaging
- Weights based on training F1 scores:
  - LightGBM: 0.632 → weight 0.51
  - XGBoost: 0.617 → weight 0.49

## Input

- LightGBM model from Phase 3
- XGBoost model from Phase 3
- Lone Wolf test data (data_features.csv)
- Ground truth labels

## Output

- Performance comparison: LightGBM vs XGBoost vs Ensemble
- Decision on best model for thesis


## Cell 1: Imports


In [35]:
import pandas as pd
import numpy as np
import pickle
import joblib
from pathlib import Path
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")


Libraries imported successfully


## Cell 2: Configuration


In [36]:
# Paths
PHASE3_DIR = Path('/Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 3 - Model Training')
TEST_DATA_DIR = Path('/Users/soni/Github/Digital-Detectives_Thesis/data/processed/Prototype Tool Output/LW')
OUTPUT_DIR = Path('/Users/soni/Github/Digital-Detectives_Thesis/notebooks/Phase 4 - Hyperparameter Tuning')

# Model paths
LIGHTGBM_MODEL = PHASE3_DIR / 'lightgbm_model.pkl'
XGBOOST_MODEL = PHASE3_DIR / 'xgboost_model.pkl'

# Test data
TEST_FEATURES = TEST_DATA_DIR / 'data_features.csv'

# Confidence threshold
CONFIDENCE_THRESHOLD = 70

print("Configuration:")
print(f"  LightGBM model: {LIGHTGBM_MODEL.exists()}")
print(f"  XGBoost model: {XGBOOST_MODEL.exists()}")
print(f"  Test data: {TEST_FEATURES.exists()}")
print(f"  Confidence threshold: {CONFIDENCE_THRESHOLD}%")


Configuration:
  LightGBM model: True
  XGBoost model: True
  Test data: True
  Confidence threshold: 70%


## Cell 3: Load Models


In [37]:
print("="*80)
print("LOADING TRAINED MODELS")
print("="*80)

# Load LightGBM
with open(LIGHTGBM_MODEL, 'rb') as f:
    lightgbm_model = pickle.load(f)
    
print(f"\n✓ LightGBM loaded")
print(f"  Type: {type(lightgbm_model).__name__}")
print(f"  Features: {lightgbm_model.n_features_in_}")

# Load XGBoost
with open(XGBOOST_MODEL, 'rb') as f:
    xgboost_model = pickle.load(f)
    
print(f"\n✓ XGBoost loaded")
print(f"  Type: {type(xgboost_model).__name__}")
print(f"  Features: {xgboost_model.n_features_in_}")


LOADING TRAINED MODELS

✓ LightGBM loaded
  Type: LGBMClassifier
  Features: 31

✓ XGBoost loaded
  Type: XGBClassifier
  Features: 31


## Cell 4: Load Test Data


In [38]:
print("="*80)
print("LOADING TEST DATA")
print("="*80)

# Load feature data
df = pd.read_csv(TEST_FEATURES, low_memory=False)

print(f"\nDataset loaded:")
print(f"  Records: {len(df):,}")
print(f"  Columns: {len(df.columns)}")

# Check for ground truth
if 'ground_truth_label' in df.columns:
    y_true = df['ground_truth_label'].fillna(0).astype(int)
    print(f"\n✓ Ground truth available")
    print(f"  Known timestomped files: {y_true.sum()}")
else:
    y_true = None
    print(f"\n⚠ No ground truth available")


LOADING TEST DATA

Dataset loaded:
  Records: 7,420
  Columns: 64

✓ Ground truth available
  Known timestomped files: 0


## Cell 5: Prepare Features (EXACT COPY FROM PROTOTYPE TOOL)


In [39]:
print("="*80)
print("PREPARING FEATURES FOR PREDICTION")
print("="*80)

# THIS IS THE EXACT CODE FROM PROTOTYPE TOOL NOTEBOOK 03 - CELL 5

print("\nCreating features to match training...")

# 1. Rename features to match training names
feature_mapping = {
    'zero_in_nanoseconds_combined': 'zero_in_nanoseconds',
    'modified_creationtime': 'creation_time_modified',
    'modified_modifiedtime': 'modified_time_modified',
    'modified_accessedtime': 'accessed_time_modified',
    'modified_mftmodifiedtime': 'mft_time_modified'
}

for old_name, new_name in feature_mapping.items():
    if old_name in df.columns:
        df[new_name] = df[old_name]
        print(f"  Mapped: {old_name} -> {new_name}")

# 2. Create missing features with default values
missing_features = {
    'cross_artifact_detected': lambda: df['has_logfile_evidence'] & df['has_usnjrnl_evidence'],
    'zero_in_nanoseconds_suspicious': lambda: False,
    'multiple_timestamps_changed': lambda: (
        df.get('creation_time_modified', False) & 
        df.get('modified_time_modified', False)
    ),
    'same_as_another_file': lambda: False,
    'is_image': lambda: df['filename'].fillna('').str.endswith(('.jpg', '.jpeg', '.png', '.gif', '.bmp'), na=False),
    'in_temp_directory': lambda: df['full_path'].fillna('').str.contains(r'temp|tmp', case=False, na=False, regex=True),
    'in_system_directory': lambda: df['full_path'].fillna('').str.contains(r'windows|system32|syswow64', case=False, na=False, regex=True),
    'in_program_files': lambda: df['full_path'].fillna('').str.contains(r'program files', case=False, na=False, regex=True),
    'has_timestamp_data': lambda: df['lf_creation_time'].notna() if 'lf_creation_time' in df.columns else False,
    'timestamp_source': lambda: (
        df['has_logfile_evidence'].astype(int) * 2 + 
        df['has_usnjrnl_evidence'].astype(int)
    )
}

for feat_name, feat_func in missing_features.items():
    if feat_name not in df.columns:
        df[feat_name] = feat_func()
        print(f"  Created: {feat_name}")

# 3. Select exact features model expects (in correct order)
expected_features = [
    'cross_artifact_detected',
    'zero_in_nanoseconds_lf',
    'zero_in_nanoseconds_suspicious',
    'zero_in_nanoseconds',
    'time_reversal_event',
    'basic_info_changed',
    'using_another_timestamp',
    'si_timestamp_changed',
    'update_resident_value',
    'creation_time_modified',
    'modified_time_modified',
    'accessed_time_modified',
    'mft_time_modified',
    'timestamp_changed_to_past',
    'multiple_timestamps_changed',
    'same_as_another_file',
    'zero_nano_time_reversal',
    'has_logfile_evidence',
    'has_usnjrnl_evidence',
    'cross_artifact_validation_score',
    'is_executable',
    'is_document',
    'is_archive',
    'is_image',
    'path_depth',
    'filename_length',
    'in_temp_directory',
    'in_system_directory',
    'in_program_files',
    'has_timestamp_data',
    'timestamp_source'
]

# Build feature matrix with exact features
X = df[expected_features].copy()

# Handle missing values
X = X.fillna(0)

# Convert boolean to int
for col in X.columns:
    if X[col].dtype == 'bool':
        X[col] = X[col].astype(int)

print(f"\nFeatures prepared:")
print(f"  Features expected by model: {len(expected_features)}")
print(f"  Features provided: {len(X.columns)}")
print(f"  Feature matrix shape: {X.shape}")

# Verify all features present
missing = set(expected_features) - set(X.columns)
if missing:
    print(f"\n⚠ WARNING: Missing features: {missing}")
else:
    print(f"\n✓ All required features present!")


PREPARING FEATURES FOR PREDICTION

Creating features to match training...
  Mapped: zero_in_nanoseconds_combined -> zero_in_nanoseconds
  Mapped: modified_creationtime -> creation_time_modified
  Mapped: modified_modifiedtime -> modified_time_modified
  Mapped: modified_accessedtime -> accessed_time_modified
  Mapped: modified_mftmodifiedtime -> mft_time_modified
  Created: cross_artifact_detected
  Created: multiple_timestamps_changed
  Created: same_as_another_file
  Created: is_image
  Created: in_temp_directory
  Created: in_system_directory
  Created: in_program_files
  Created: has_timestamp_data
  Created: timestamp_source

Features prepared:
  Features expected by model: 31
  Features provided: 31
  Feature matrix shape: (7420, 31)

✓ All required features present!


## Cell 6: Test Individual Models


In [40]:
print("="*80)
print("TESTING INDIVIDUAL MODELS")
print("="*80)

# LightGBM predictions
lgbm_proba = lightgbm_model.predict_proba(X)[:, 1]
lgbm_confidence = lgbm_proba * 100
lgbm_flagged = lgbm_confidence >= CONFIDENCE_THRESHOLD

print(f"\nLightGBM:")
print(f"  Max confidence: {lgbm_confidence.max():.2f}%")
print(f"  Mean confidence: {lgbm_confidence.mean():.2f}%")
print(f"  Files flagged: {lgbm_flagged.sum()}")

# XGBoost predictions
xgb_proba = xgboost_model.predict_proba(X)[:, 1]
xgb_confidence = xgb_proba * 100
xgb_flagged = xgb_confidence >= CONFIDENCE_THRESHOLD

print(f"\nXGBoost:")
print(f"  Max confidence: {xgb_confidence.max():.2f}%")
print(f"  Mean confidence: {xgb_confidence.mean():.2f}%")
print(f"  Files flagged: {xgb_flagged.sum()}")

# Check if models are working
if lgbm_flagged.sum() == 0 and xgb_flagged.sum() == 0:
    print(f"\n⚠ WARNING: Neither model detected any files!")
    print(f"\nDiagnostic - Top 10 predictions:")
    
    diagnostic_df = pd.DataFrame({
        'filename': df['filename'],
        'lgbm_conf': lgbm_confidence,
        'xgb_conf': xgb_confidence,
        'ground_truth': y_true if y_true is not None else 0
    })
    print(diagnostic_df.nlargest(10, 'lgbm_conf')[['filename', 'lgbm_conf', 'xgb_conf', 'ground_truth']])
else:
    print(f"\n✓ Models are generating detections")


TESTING INDIVIDUAL MODELS

LightGBM:
  Max confidence: 99.84%
  Mean confidence: 1.72%
  Files flagged: 24

XGBoost:
  Max confidence: 99.62%
  Mean confidence: 2.17%
  Files flagged: 24

✓ Models are generating detections


## Cell 7: Create Ensemble


In [41]:
print("="*80)
print("CREATING ENSEMBLE MODEL")
print("="*80)

# Training F1 scores from Phase 3
lightgbm_f1 = 0.632
xgboost_f1 = 0.617

# Calculate weights
total_f1 = lightgbm_f1 + xgboost_f1
lgbm_weight = lightgbm_f1 / total_f1
xgb_weight = xgboost_f1 / total_f1

print(f"\nWeights based on training F1:")
print(f"  LightGBM (F1={lightgbm_f1:.3f}): {lgbm_weight:.3f}")
print(f"  XGBoost  (F1={xgboost_f1:.3f}): {xgb_weight:.3f}")

# Create ensemble
ensemble = VotingClassifier(
    estimators=[
        ('lightgbm', lightgbm_model),
        ('xgboost', xgboost_model)
    ],
    voting='soft',
    weights=[lgbm_weight, xgb_weight]
)

# Note: VotingClassifier needs fit() to be called, but we're using pre-trained models
# So we'll manually calculate weighted average

ensemble_proba = (lgbm_proba * lgbm_weight + xgb_proba * xgb_weight)
ensemble_confidence = ensemble_proba * 100
ensemble_flagged = ensemble_confidence >= CONFIDENCE_THRESHOLD

print(f"\n✓ Ensemble created (weighted soft voting)")
print(f"\nEnsemble:")
print(f"  Max confidence: {ensemble_confidence.max():.2f}%")
print(f"  Mean confidence: {ensemble_confidence.mean():.2f}%")
print(f"  Files flagged: {ensemble_flagged.sum()}")


CREATING ENSEMBLE MODEL

Weights based on training F1:
  LightGBM (F1=0.632): 0.506
  XGBoost  (F1=0.617): 0.494

✓ Ensemble created (weighted soft voting)

Ensemble:
  Max confidence: 99.63%
  Mean confidence: 1.94%
  Files flagged: 24


## Cell 8: Performance Comparison


In [42]:
print("="*80)
print("PERFORMANCE COMPARISON")
print("="*80)

if y_true is not None:
    # Calculate metrics for each model
    models = {
        'LightGBM': lgbm_flagged,
        'XGBoost': xgb_flagged,
        'Ensemble': ensemble_flagged
    }
    
    results = []
    
    for model_name, predictions in models.items():
        if predictions.sum() > 0:
            precision = precision_score(y_true, predictions, zero_division=0)
            recall = recall_score(y_true, predictions, zero_division=0)
            f1 = f1_score(y_true, predictions, zero_division=0)
            
            tp = ((predictions == 1) & (y_true == 1)).sum()
            fp = ((predictions == 1) & (y_true == 0)).sum()
            fn = ((predictions == 0) & (y_true == 1)).sum()
        else:
            precision = recall = f1 = 0
            tp = fp = 0
            fn = y_true.sum()
        
        results.append({
            'Model': model_name,
            'Flagged': int(predictions.sum()),
            'TP': int(tp),
            'FP': int(fp),
            'FN': int(fn),
            'Precision': precision,
            'Recall': recall,
            'F1': f1
        })
    
    results_df = pd.DataFrame(results)
    
    print(f"\nGround Truth: {int(y_true.sum())} known timestomped files\n")
    print(results_df.to_string(index=False))
    
    # Determine best model
    best_idx = results_df['F1'].idxmax()
    best_model = results_df.loc[best_idx, 'Model']
    best_f1 = results_df.loc[best_idx, 'F1']
    
    print(f"\n" + "="*80)
    print(f"BEST MODEL: {best_model} (F1 = {best_f1:.4f})")
    print("="*80)
    
else:
    print("\n⚠ Cannot calculate performance - no ground truth available")
    print("\nDetection counts only:")
    print(f"  LightGBM: {lgbm_flagged.sum()}")
    print(f"  XGBoost: {xgb_flagged.sum()}")
    print(f"  Ensemble: {ensemble_flagged.sum()}")


PERFORMANCE COMPARISON

Ground Truth: 0 known timestomped files

   Model  Flagged  TP  FP  FN  Precision  Recall  F1
LightGBM       24   0  24   0        0.0     0.0 0.0
 XGBoost       24   0  24   0        0.0     0.0 0.0
Ensemble       24   0  24   0        0.0     0.0 0.0

BEST MODEL: LightGBM (F1 = 0.0000)


## Cell 9: Save Results


In [43]:
print("="*80)
print("SAVING RESULTS")
print("="*80)

# Create output dataframe
output_df = df[['filename', 'full_path']].copy()
output_df['lightgbm_confidence'] = lgbm_confidence
output_df['xgboost_confidence'] = xgb_confidence
output_df['ensemble_confidence'] = ensemble_confidence
output_df['lightgbm_flagged'] = lgbm_flagged
output_df['xgboost_flagged'] = xgb_flagged
output_df['ensemble_flagged'] = ensemble_flagged

if y_true is not None:
    output_df['ground_truth'] = y_true

# Save all predictions
output_path = OUTPUT_DIR / 'ensemble_predictions.csv'
output_df.to_csv(output_path, index=False)
print(f"\n✓ Saved: {output_path}")

# Save flagged files from best model
if y_true is not None and 'best_model' in locals():
    if best_model == 'Ensemble':
        flagged_df = output_df[ensemble_flagged].copy()
    elif best_model == 'XGBoost':
        flagged_df = output_df[xgb_flagged].copy()
    else:
        flagged_df = output_df[lgbm_flagged].copy()
    
    flagged_path = OUTPUT_DIR / f'ensemble_{best_model.lower()}_flagged.csv'
    flagged_df.to_csv(flagged_path, index=False)
    print(f"✓ Saved best model ({best_model}) flagged files: {flagged_path}")

print(f"\n✓ All results saved")


SAVING RESULTS

✓ Saved: /Users/soni/Github/Digital-Detectives_Thesis/notebooks/Phase 4 - Hyperparameter Tuning/ensemble_predictions.csv
✓ Saved best model (LightGBM) flagged files: /Users/soni/Github/Digital-Detectives_Thesis/notebooks/Phase 4 - Hyperparameter Tuning/ensemble_lightgbm_flagged.csv

✓ All results saved


## Cell 10: Final Recommendation


In [44]:
print("="*80)
print("FINAL RECOMMENDATION FOR THESIS")
print("="*80)

if y_true is not None and 'results_df' in locals():
    print(f"\nBased on Lone Wolf test set performance:\n")
    
    for _, row in results_df.iterrows():
        print(f"{row['Model']}:")
        print(f"  Recall: {row['Recall']*100:.1f}% ({row['TP']}/{int(y_true.sum())} known files detected)")
        print(f"  Precision: {row['Precision']*100:.1f}% ({row['TP']}/{row['Flagged']} flagged are true positives)")
        print(f"  F1-Score: {row['F1']:.4f}")
        print()
    
    # Check if ensemble improved
    lgbm_f1 = results_df[results_df['Model'] == 'LightGBM']['F1'].values[0]
    ensemble_f1 = results_df[results_df['Model'] == 'Ensemble']['F1'].values[0]
    
    improvement = ensemble_f1 - lgbm_f1
    
    print("="*80)
    if improvement > 0.01:
        print("RECOMMENDATION: Use ENSEMBLE model")
        print(f"  → Ensemble improves F1 by {improvement:.4f} over LightGBM alone")
    elif improvement > 0:
        print("RECOMMENDATION: Use LightGBM (single model)")
        print(f"  → Ensemble improvement ({improvement:.4f}) is marginal")
        print(f"  → Single model is simpler for deployment and interpretation")
    else:
        print("RECOMMENDATION: Use LightGBM (single model)")
        print(f"  → Ensemble does not improve performance")
        print(f"  → Single model is simpler and performs equally well")
    print("="*80)
    
else:
    print("\nCannot make recommendation - no validation data available")

print("\n✓ Ensemble analysis complete")


FINAL RECOMMENDATION FOR THESIS

Based on Lone Wolf test set performance:

LightGBM:
  Recall: 0.0% (0/0 known files detected)
  Precision: 0.0% (0/24 flagged are true positives)
  F1-Score: 0.0000

XGBoost:
  Recall: 0.0% (0/0 known files detected)
  Precision: 0.0% (0/24 flagged are true positives)
  F1-Score: 0.0000

Ensemble:
  Recall: 0.0% (0/0 known files detected)
  Precision: 0.0% (0/24 flagged are true positives)
  F1-Score: 0.0000

RECOMMENDATION: Use LightGBM (single model)
  → Ensemble does not improve performance
  → Single model is simpler and performs equally well

✓ Ensemble analysis complete
